In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
candle_data_path = "nifty_1min_full.json"
file_path = Path(f"assets/historical/{candle_data_path}")
nifty_all = pd.read_json(file_path, lines=True)
date_ = '2026-04-16' # YYYY-MM-DD

In [3]:
import pandas as pd
import json

def parse_date(x):
    try:
        if isinstance(x, dict):
            return x['date']
        if isinstance(x, str):
            return json.loads(x.rstrip('}') + '}')['date']
        return x
    except:
        return None

nifty_all['date'] = (
    pd.to_datetime(nifty_all['date'].map(parse_date), unit='ms', errors='coerce', utc=True)
    .dt.tz_convert('Asia/Kolkata')
)

day_df = nifty_all.loc[
    nifty_all['date'].dt.date == pd.to_datetime(date_).date()
]

In [4]:
day_df

,date,open,high,low,close,volume
1160092,2026-04-16 09:15:00+05:30,24385.20,24392.35,24333.95,24355.15,0
1160093,2026-04-16 09:16:00+05:30,24353.50,24367.60,24341.90,24360.40,0
1160094,2026-04-16 09:17:00+05:30,24360.90,24381.10,24360.90,24380.35,0
1160095,2026-04-16 09:18:00+05:30,24379.25,24379.25,24364.60,24376.20,0
1160096,2026-04-16 09:19:00+05:30,24373.05,24390.55,24373.05,24390.55,0
...,...,...,...,...,...,...
1743508,2026-04-16 15:25:00+05:30,24196.10,24199.95,24189.20,24193.15,0
1743509,2026-04-16 15:26:00+05:30,24193.15,24196.65,24190.10,24194.05,0
1743510,2026-04-16 15:27:00+05:30,24194.05,24195.40,24191.00,24192.80,0
1743511,2026-04-16 15:28:00+05:30,24192.80,24194.70,24185.15,24187.50,0


In [5]:
day_df.shape

(750, 6)

In [6]:
import pandas as pd
import numpy as np
import pandas_ta as ta

def tradingview_bb(close, length=20, mult=2.0):
    basis = close.rolling(length).mean()
    std = close.rolling(length).std(ddof=0)  # FIX
    dev = mult * std
    upper = basis + dev
    lower = basis - dev
    return basis, upper, lower

def tradingview_roc(close, length=10):
    close = close.astype(float)

    prev = close.shift(length)

    # EXACT order like TradingView
    roc = 100 * (close / prev - 1)

    return roc
def compute_signals(df,
                    bb_length=20,
                    bb_mult=2.0,
                    dmi_length=14,
                    roc_length=10):

    df = df.copy()

    # Ensure sorted (VERY IMPORTANT)
    df = df.sort_values('date').reset_index(drop=True)

    # -----------------------
    # Bollinger Bands
    # -----------------------
    bb = ta.bbands(df['close'], length=bb_length, std=bb_mult)

    # Avoid column name issues → use iloc
    # df['lower_bb'] = bb.iloc[:, 0]
    # df['basis']    = bb.iloc[:, 1]
    # df['upper_bb'] = bb.iloc[:, 2]
    df['basis'], df['upper_bb'], df['lower_bb'] = tradingview_bb(df['close'], bb_length, bb_mult)

    # -----------------------
    # DMI / ADX
    # -----------------------
    dmi = ta.adx(df['high'], df['low'], df['close'], length=dmi_length)

    # Robust column extraction
    df['plus_di']  = dmi[[col for col in dmi.columns if "DMP" in col][0]]
    df['minus_di'] = dmi[[col for col in dmi.columns if "DMN" in col][0]]
    df['adx']      = dmi[[col for col in dmi.columns if "ADX" in col][0]]

    # -----------------------
    # ROC (convert % → decimal like TradingView)
    # -----------------------
    # df['roc'] = ta.roc(df['close'], length=roc_length) / 100.0
    df['roc'] = tradingview_roc(df['close'], length=roc_length)

    # -----------------------
    # Signal Logic (EXACT Pine translation)
    # -----------------------
    df['long_signal'] = (
        (df['close'].shift(1) < df['lower_bb'].shift(1)) &
        (df['open'] < df['lower_bb']) &
        (df['minus_di'] < 0.95 * df['minus_di'].shift(1)) &
        (df['roc'] > df['roc'].shift(1) + 0.01)
    )

    df['short_signal'] = (
        (df['close'].shift(1) > df['upper_bb'].shift(1)) &
        (df['open'] > df['upper_bb']) &
        (df['plus_di'] < 0.95 * df['plus_di'].shift(1)) &
        (df['roc'] < df['roc'].shift(1) - 0.01)
    )

    # -----------------------
    # Final Signal Column
    # -----------------------
    df['signal'] = np.where(df['long_signal'], 'BUY',
                    np.where(df['short_signal'], 'SELL', None))

    return df

In [7]:
day_df = compute_signals(day_df)

In [8]:
print(day_df[['date', 'close', 'signal']].tail(20))

                         date     close signal
730 2026-04-16 15:20:00+05:30  24195.35    NaN
731 2026-04-16 15:20:00+05:30  24195.35    NaN
732 2026-04-16 15:21:00+05:30  24194.80    NaN
733 2026-04-16 15:21:00+05:30  24194.80    NaN
734 2026-04-16 15:22:00+05:30  24196.20    NaN
735 2026-04-16 15:22:00+05:30  24196.20    NaN
736 2026-04-16 15:23:00+05:30  24195.40    NaN
737 2026-04-16 15:23:00+05:30  24195.40    NaN
738 2026-04-16 15:24:00+05:30  24196.00    NaN
739 2026-04-16 15:24:00+05:30  24196.00    NaN
740 2026-04-16 15:25:00+05:30  24193.15    NaN
741 2026-04-16 15:25:00+05:30  24193.15    NaN
742 2026-04-16 15:26:00+05:30  24194.05    NaN
743 2026-04-16 15:26:00+05:30  24194.05    NaN
744 2026-04-16 15:27:00+05:30  24192.80    NaN
745 2026-04-16 15:27:00+05:30  24192.80    NaN
746 2026-04-16 15:28:00+05:30  24187.50    NaN
747 2026-04-16 15:28:00+05:30  24187.50    NaN
748 2026-04-16 15:29:00+05:30  24188.40    NaN
749 2026-04-16 15:29:00+05:30  24188.40    NaN


In [9]:
day_df.tail(5)

,date,open,high,low,close,volume,basis,upper_bb,lower_bb,plus_di,minus_di,adx,roc,long_signal,short_signal,signal
745,2026-04-16 15:27:00+05:30,24194.05,24195.40,24191.00,24192.8,0,24191.745,24203.858253,24179.631747,8.406437,9.169504,11.044230,-0.014052,False,False,NaN
746,2026-04-16 15:28:00+05:30,24192.80,24194.70,24185.15,24187.5,0,24192.175,24202.972986,24181.377014,7.782665,13.034449,12.057371,-0.032651,False,False,NaN
747,2026-04-16 15:28:00+05:30,24192.80,24194.70,24185.15,24187.5,0,24192.605,24201.823617,24183.386383,7.206776,12.069946,12.998144,-0.032651,False,False,NaN
748,2026-04-16 15:29:00+05:30,24187.50,24190.45,24178.85,24188.4,0,24192.985,24200.735490,24185.234510,6.570763,15.797749,15.016126,-0.031410,False,False,NaN
749,2026-04-16 15:29:00+05:30,24187.50,24190.45,24178.85,24188.4,0,24193.365,24199.196132,24187.533868,6.000473,14.426632,16.889966,-0.031410,False,False,NaN


In [10]:
day_df["short_signal"].value_counts()

short_signal
False    748
True       2
Name: count, dtype: int64

In [11]:
day_df["long_signal"].value_counts()

long_signal
False    748
True       2
Name: count, dtype: int64

In [12]:
day_df["signal"].value_counts()

signal
BUY     2
SELL    2
Name: count, dtype: int64

## AGG

In [13]:
day_df[day_df["signal"]=="BUY"]

,date,open,high,low,close,volume,basis,upper_bb,lower_bb,plus_di,minus_di,adx,roc,long_signal,short_signal,signal
204,2026-04-16 10:57:00+05:30,24303.25,24314.05,24302.15,24312.8,0,24314.3175,24322.787463,24305.847537,8.248723,13.291902,18.717729,-0.005347,True,False,BUY
694,2026-04-16 15:02:00+05:30,24191.70,24217.55,24191.00,24210.9,0,24222.9500,24251.057597,24194.842403,9.688607,16.919347,31.069726,-0.025602,True,False,BUY


In [14]:
day_df[day_df["signal"]=="SELL"]

,date,open,high,low,close,volume,basis,upper_bb,lower_bb,plus_di,minus_di,adx,roc,long_signal,short_signal,signal
490,2026-04-16 13:20:00+05:30,24137.85,24140.70,24130.75,24132.85,0,24124.4300,24137.557163,24111.302837,12.660537,10.548320,23.436221,0.034405,False,True,SELL
552,2026-04-16 13:51:00+05:30,24178.85,24182.35,24167.70,24169.35,0,24148.1175,24175.442548,24120.792452,26.923042,2.261101,53.817393,0.082611,False,True,SELL
